# The Complete Guide to Polarimetry: Theory and Measurement

## 1. What are we measuring? (The Goal)
In polarimetry, we aim to define the **Polarization State** of a light source. Imagine a long skipping rope tied to a wall:
* **Total Intensity ($I$):** How hard are you shaking the rope? (Total Energy).
* **Orientation Angle ($\psi$):** In what direction are you shaking it? (Up-down, side-side, or diagonal).
* **Degree of Linear Polarization ($DoLP$):** Is everyone shaking the rope in the same direction (100% polarized) or is it a chaotic mess of random shakes (0% polarized)?

---

## 2. The Physics of Birefringence
When light enters a birefringent crystal (like Calcite), it splits into two rays that travel at different speeds:

| Ray Type | Name | Behavior |
| :--- | :--- | :--- |
| **O-Ray** | **Ordinary Ray** | Follows standard refraction ($n_o$). It sees the crystal as a normal material. |
| **E-Ray** | **Extraordinary Ray** | Travels at a different speed ($n_e$). It sees a different electrical environment. |



**Phase Delay:** Because the E-ray and O-ray travel at different speeds, one gets "delayed" relative to the other. This delay is the fundamental principle used to manipulate light in a lab.

---

## 3. The Experimental Tools

### The Half-Wave Plate (HWP)
A HWP introduces a phase delay of $180^\circ$ ($\pi$). It acts as a **Polarization Mirror**.
* **The $2\theta$ Rule:** If you rotate the HWP by an angle $\theta$, the polarization vector of the light rotates by **$2\theta$**. 
* **Intuition:** Rotating the HWP by $22.5^\circ$ causes the light's polarization to flip $45^\circ$.



### The Wollaston Prism
This is a specialized prism that physically separates the O-ray and E-ray into two diverging beams. 
* **Benefit:** It allows us to measure both polarization components simultaneously without blocking or losing any light.

---

## 4. The Stokes Parameters ($I, Q, U, V$)
The Stokes vector is the "Identity Card" of a light beam, defined by intensity measurements ($P$):

* **$I$ (Total Intensity):** $P_{Horizontal} + P_{Vertical}$. The total brightness.
* **$Q$ (Horizontal vs. Vertical):** $P_{H} - P_{V}$. Positive = Horizontal; Negative = Vertical.
* **$U$ (Diagonals):** $P_{+45^\circ} - P_{45^\circ}$. Tells us if the light "leans" diagonally.
* **$V$ (Circular):** $P_{Right} - P_{Left}$. Tells us if the light is "corkscrewing" (Circularly polarized).



---

## 5. Key Calculations

### Orientation Angle ($\psi$)
The "Compass Heading" of the light. It tells you the physical angle of the electric field oscillation.
$$\psi = \frac{1}{2} \arctan\left(\frac{U}{Q}\right)$$

### Degree of Linear Polarization ($DoLP$)
This describes what fraction of the light is ordered vs. unpolarized.
$$DoLP = \frac{\sqrt{Q^2 + U^2}}{I}$$
* **$DoLP = 1.0$:** Perfectly ordered (e.g., a laser).
* **$DoLP = 0.0$:** Completely random (e.g., a lightbulb).

---

## 6. How the Polarimeter Works
1.  **Modulate:** We spin a **Half-Wave Plate** to rotate the incoming light's polarization.
2.  **Split:** A **Wollaston Prism** separates the light into two channels (A and B).
3.  **Analyze:** As the HWP spins, the intensities in A and B create a sine wave. 
4.  **Normalize:** We calculate the ratio $\frac{A - B}{A + B}$. This cancels out noise from source flickering or atmospheric changes.

In [1]:
import sys
print(sys.executable)
print(sys.version)

import numpy
import scipy
import matplotlib
import pandas
import astropy

print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("pandas:", pandas.__version__)
print("astropy:", astropy.__version__)

/usr/local/python/3.12.1/bin/python
3.12.1 (main, Nov 27 2025, 10:47:52) [GCC 13.3.0]
numpy: 2.4.3
scipy: 1.17.1
matplotlib: 3.10.8
pandas: 3.0.1
astropy: 7.2.0


In [2]:
# Core imports

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from astropy.visualization import simple_norm
from astropy.table import Table

from scipy.ndimage import shift as ndi_shift
from scipy.spatial.distance import cdist

from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from photutils.centroids import centroid_2dg, centroid_com

import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["image.origin"] = "lower"

# Set data paths

In this cell we define where the raw data live.

We use `pathlib.Path` instead of plain strings because it is safer and cleaner for file handling.

For example:

- `Path("data") / "bias"` makes a proper file path
- it works more robustly across platforms
- it avoids many manual slash mistakes

You should edit the path below to match your actual folder.

In [ ]:
# Edit this to your actual dataset folder
DATA_DIR = Path("data")

BIAS_DIR = DATA_DIR / "bias"
SCI_DIR  = DATA_DIR / "science"
FLAT_DIR = DATA_DIR / "flat"   # optional, if flats exist

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("DATA_DIR :", DATA_DIR.resolve())
print("BIAS_DIR :", BIAS_DIR.resolve())
print("SCI_DIR  :", SCI_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

# Helper functions for FITS handling

This cell defines small reusable functions.

We use functions so the notebook stays modular and readable.

## Functions introduced

### `load_fits_data(path)`
Reads the primary image array and header from a FITS file.

### `show_image(data, title="", percentile=99)`
Displays an image with robust normalization.

Instead of scaling from absolute min to max, we often use percentile based normalization because CCD images may contain:
- cosmic rays
- hot pixels
- saturated stars

If we scale using absolute max, the whole image may look black except one bright point.

### `get_hwp_angle(header, filename=None)`
Tries to read the HWP angle from the FITS header.
If not found, it can later be adapted to parse the filename.

This part is instrument dependent.

In [ ]:
def load_fits_data(path):
    """
    Load FITS image and header.

    Parameters
    ----------
    path : str or Path
        Path to FITS file.

    Returns
    -------
    data : 2D numpy array
        Image data converted to float.
    header : astropy.io.fits.Header
        FITS header.
    """
    with fits.open(path) as hdul:
        data = hdul[0].data.astype(float)
        header = hdul[0].header
    return data, header


def show_image(data, title="", percentile=99.5, cmap="gray"):
    """
    Display image with percentile based normalization.

    Percentile normalization is useful because astronomical images often
    contain a few very bright pixels or stars that would otherwise dominate
    the display scale.

    Parameters
    ----------
    data : 2D array
        Image data.
    title : str
        Plot title.
    percentile : float
        Upper percentile for display normalization.
    cmap : str
        Colormap.
    """
    norm = simple_norm(data, stretch="linear", percent=percentile)
    plt.figure(figsize=(7, 7))
    plt.imshow(data, norm=norm, cmap=cmap)
    plt.colorbar(label="Counts")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def get_hwp_angle(header, filename=None):
    """
    Try to extract HWP angle from FITS header.

    You MUST adapt this function once you inspect your real FITS headers.

    Common possible keywords:
    - HWP
    - HWPANG
    - RETANG
    - WPLATE
    - ANGLE

    Returns
    -------
    angle : float or None
    """
    candidate_keys = ["HWP", "HWPANG", "RETANG", "ANGLE", "WPLATE"]
    for key in candidate_keys:
        if key in header:
            try:
                return float(header[key])
            except Exception:
                pass

    # optional fallback from filename can be added here
    return None